# 专家系统实验——《推理王国》第2章

本 notebook 构建了一个判断蔷薇科花卉的微型专家系统，用**前向链接（Forward Chaining）推理引擎**实现。  
通过四个案例，亲手复现书中描述的**封闭世界假设（CWA）**的三种不同失效模式。

| 案例 | 类型 | 结果 | 失效模式 |
|---|---|---|---|
| A | 正常案例 | 正确 | 无 |
| B | 边界案例 | 两个结论，无置信度 | 危险的沉默失败 |
| C | 分布外案例 | 自信的错误答案 | CWA：矛盾特征被静默丢弃 |
| D | 规则盲区案例 | 空结论 | CWA：另一种失效——沉默的无结论 |

In [38]:
# 规则的数据结构：每条规则是一个字典
# 'conditions'：前提条件列表（列表中所有条件都满足才能触发）
# 'conclusion'：规则结论（一个字符串）
# 'name'：规则名称，便于追踪推理轨迹

# # 示例：以"判断菜系"为领域的规则库
# rules = [
#     # ── 中间层规则：识别食材特征 ──────────────────────────────
#     {"name": "R1",  "conditions": ["有鱼腥草"],            "conclusion": "使用了四川特色食材"},
#     {"name": "R2",  "conditions": ["有辣椒", "有花椒"],    "conclusion": "麻辣风格"},
#     {"name": "R3",  "conditions": ["有生鱼片"],            "conclusion": "使用了日式食材"},
#     {"name": "R4",  "conditions": ["有味噌"],              "conclusion": "使用了日式食材"},
#     {"name": "R5",  "conditions": ["有豆豉", "有蒸鱼豉油"],"conclusion": "粤式调味"},
#     {"name": "R6",  "conditions": ["有姜葱"],              "conclusion": "粤式底味"},

#     # ── 中间层规则：识别烹饪方式 ──────────────────────────────
#     {"name": "R7",  "conditions": ["大火爆炒"],            "conclusion": "高温快炒风格"},
#     {"name": "R8",  "conditions": ["清蒸"],                "conclusion": "清淡烹饪风格"},
#     {"name": "R9",  "conditions": ["炭火烤制"],            "conclusion": "烧烤风格"},

#     # ── 终结层规则：得出菜系结论 ──────────────────────────────
#     {"name": "R10", "conditions": ["使用了四川特色食材", "麻辣风格"], "conclusion": "是川菜"},
#     {"name": "R11", "conditions": ["使用了日式食材"],      "conclusion": "是日料"},
#     {"name": "R12", "conditions": ["粤式调味", "清淡烹饪风格"], "conclusion": "是粤菜"},
#     {"name": "R13", "conditions": ["粤式底味", "粤式调味"],"conclusion": "是粤菜"},
#     {"name": "R14", "conditions": ["烧烤风格", "使用了四川特色食材"], "conclusion": "是川式烧烤"},
#     {"name": "R15", "conditions": ["高温快炒风格", "粤式调味"], "conclusion": "是粤式小炒"},
# ]

# 判断蔷薇科的花
rules = [
    # ── 中间层规则：识别花朵特征 ──────────────────────────────
    {"name": "R1",  "conditions": ["花瓣是白色"],              "conclusion": "白色花瓣"},
    {"name": "R2",  "conditions": ["花瓣颜色和花苞不同"],  "conclusion": "花瓣颜色会变"},
    {"name": "R3",  "conditions": ["花瓣有很多层"],    "conclusion": "重瓣"},
    {"name": "R4",  "conditions": ["花瓣圆滑"],            "conclusion": "圆形花瓣"},
    {"name": "R5",  "conditions": ["花瓣偏尖"],              "conclusion": "尖形花瓣"},
    {"name": "R6",  "conditions": ["花瓣有缺口"],           "conclusion": "缺口花瓣"},

    # ── 中间层规则：识别生长方式 ──────────────────────────────
    {"name": "R7",  "conditions": ["花朵直接开在树枝上"],            "conclusion": "贴枝开放"},
    {"name": "R8",  "conditions": ["花朵聚集成一簇"],                "conclusion": "成簇开放"},
    {"name": "R9",  "conditions": ["在3月中旬之前开放"],            "conclusion": "早春开放"},

    # ── 终结层规则：得出菜系结论 ──────────────────────────────
    {"name": "R10", "conditions": ["白色花瓣", "成簇开放", "圆形花瓣"], "conclusion": "是梨花"},
    {"name": "R11", "conditions": ["缺口花瓣", "成簇开放"],      "conclusion": "是樱花"},
    {"name": "R12", "conditions": ["白色花瓣", "圆形花瓣","贴枝开放"], "conclusion": "是杏花"},
    {"name": "R13", "conditions": ["早春开放", "圆形花瓣","贴枝开放"],"conclusion": "是梅花"},
    {"name": "R14", "conditions": ["尖形花瓣", "贴枝开放"], "conclusion": "是桃花"}
]



In [39]:
def forward_chain(rules, initial_facts):
    """
    前向链接推理引擎。
    rules: 规则列表，每条规则有 'conditions'、'conclusion'、'name' 三个字段。
    initial_facts: 初始观测事实的集合（set）。
    返回：(最终事实集合, 推理轨迹列表)
    """
    known_facts = set(initial_facts)
    reasoning_trace = []

    while True:
        new_fact_added = False
        for rule in rules:
            conditions_met = all(cond in known_facts for cond in rule["conditions"])
            if conditions_met and rule["conclusion"] not in known_facts:
                known_facts.add(rule["conclusion"])
                reasoning_trace.append(
                    f"触发 {rule['name']}: {rule['conditions']} → {rule['conclusion']}"
                )
                new_fact_added = True
        if not new_fact_added:
            break

    return known_facts, reasoning_trace


def print_results(facts, trace, title=""):
    """统一格式输出推理轨迹和最终结论。"""
    if title:
        print(f"=== {title} ===")
    print("推理轨迹：")
    for step in trace:
        print(" ", step)
    conclusions = [f for f in facts if f.startswith("是")]
    print(f"\n最终结论：{'、'.join(conclusions) if conclusions else '（无结论——规则覆盖盲区）'}\n")


# ── 案例 A：正常案例 ────────────────────────────────────────────
# 梨花标准特征：白色花瓣 + 圆形花瓣 + 成簇开放
initial_A = {"花瓣是白色", "花瓣圆滑", "花朵聚集成一簇"}
facts_A, trace_A = forward_chain(rules, initial_A)
print_results(facts_A, trace_A, "案例 A：正常案例（梨花）")

=== 案例 A：正常案例（梨花） ===
推理轨迹：
  触发 R1: ['花瓣是白色'] → 白色花瓣
  触发 R4: ['花瓣圆滑'] → 圆形花瓣
  触发 R8: ['花朵聚集成一簇'] → 成簇开放
  触发 R10: ['白色花瓣', '成簇开放', '圆形花瓣'] → 是梨花

最终结论：是梨花



**案例 A 分析**：系统正常工作。输入特征完整匹配 R10（白色花瓣 + 成簇开放 + 圆形花瓣），推理链条清晰可追溯——这是专家系统最大的优势：**透明度**。每一步结论都有对应的规则来源，人类可以逐条复核。

这也是 MYCIN 在 1979 年盲测中胜出的原因：在规则库覆盖良好的情况下，系统表现超过部分专家，且推理过程完全可解释。

In [40]:
# ── 案例 B：边界案例 ────────────────────────────────────────────
# 同时满足梨花和杏花前提条件的模糊输入：
# R10 梨花需要：白色花瓣 + 成簇开放 + 圆形花瓣
# R12 杏花需要：白色花瓣 + 圆形花瓣 + 贴枝开放
# → 同时输入"成簇开放"和"贴枝开放"，两条终结规则都被满足
initial_B = {"花瓣是白色", "花瓣圆滑", "花朵直接开在树枝上", "花朵聚集成一簇"}
facts_B, trace_B = forward_chain(rules, initial_B)
print_results(facts_B, trace_B, "案例 B：边界案例（梨花 + 杏花）")

=== 案例 B：边界案例（梨花 + 杏花） ===
推理轨迹：
  触发 R1: ['花瓣是白色'] → 白色花瓣
  触发 R4: ['花瓣圆滑'] → 圆形花瓣
  触发 R7: ['花朵直接开在树枝上'] → 贴枝开放
  触发 R8: ['花朵聚集成一簇'] → 成簇开放
  触发 R10: ['白色花瓣', '成簇开放', '圆形花瓣'] → 是梨花
  触发 R12: ['白色花瓣', '圆形花瓣', '贴枝开放'] → 是杏花

最终结论：是杏花、是梨花



**案例 B 分析：危险的沉默失败**

系统同时报告了"是梨花"和"是杏花"两个结论——这比 MYCIN 的猎豹/老虎案例稍好，至少没有隐藏掉一个答案。但这仍然是一种**危险的失败**，原因不在于答案错了，而在于：

- 系统对两个结论**一视同仁**，没有任何置信度区分。用户无法知道哪个更可能是对的。
- 系统没有提示用户"这是一个模糊边界情况，需要额外信息来区分"，它只是堆出两个结论，表现得好像这是正常行为。
- **用户不知道自己需要追问**——这才是"危险"的根源。

理想的系统应该给出类似：*"梨花可能性 65%，杏花可能性 35%，建议确认花朵是否同时具备贴枝和成簇特征"*。  
这正是贝叶斯/概率方法相比符号规则系统的根本优势：输出**分布**而非**点答案**，让不确定性对用户可见。

In [41]:
# ── 案例 C：分布外案例 ────────────────────────────────────────────
# 假想杂交物种：同时具备圆形花瓣（梨花特征）和缺口花瓣（樱花特征）
# 这是规则库从未见过的组合——两种特征在现实中相互矛盾
initial_C = {"花瓣有缺口", "花瓣圆滑", "花朵聚集成一簇"}
facts_C, trace_C = forward_chain(rules, initial_C)
print_results(facts_C, trace_C, "案例 C：分布外案例（杂交物种）")

=== 案例 C：分布外案例（杂交物种） ===
推理轨迹：
  触发 R4: ['花瓣圆滑'] → 圆形花瓣
  触发 R6: ['花瓣有缺口'] → 缺口花瓣
  触发 R8: ['花朵聚集成一簇'] → 成簇开放
  触发 R11: ['缺口花瓣', '成簇开放'] → 是樱花

最终结论：是樱花



**案例 C 分析：自信的错误答案——CWA 的核心危险**

推理轨迹触发了 R4（圆形花瓣）、R6（缺口花瓣）、R8（成簇开放），最终由 R11 得出"是樱花"。

**问题出在哪里？**

R11 的设计是一条**纯正向规则**：
```
缺口花瓣 AND 成簇开放 → 是樱花
```
它只描述了"满足什么条件就是樱花"，**没有描述"什么特征与樱花不相容"**。  
所以当输入同时携带了互相矛盾的信息——"圆形花瓣"（梨花的特征）和"缺口花瓣"（樱花的特征）——系统对这个矛盾视而不见，只取对 R11 有用的证据（缺口花瓣 + 成簇开放），静默地丢弃了"圆形花瓣"这条反驳信息。

**这正是封闭世界假设（CWA）的具体缺陷**：

> 规则库里没有"圆形花瓣 + 缺口花瓣同时出现是矛盾"这条知识，  
> 所以在 CWA 下，这个矛盾"不存在"，系统继续推理，得出一个自信的错误结论。

系统不困惑，不报警，不说"我没见过这种组合"。它只是给出"是樱花"——而且语气和案例 A 一模一样。  
书中原话："不会困惑，只会崩溃，而且崩溃得很自信。"

In [44]:
# ── 案例 D：规则盲区案例 ────────────────────────────────────────
# 尖形花瓣 + 成簇开放 + 早春开放
# 最接近的规则：
#   R14 桃花需要 尖形花瓣 + 贴枝开放（这朵是成簇，不是贴枝）
#   R11 樱花需要 缺口花瓣 + 成簇开放（这朵是尖形，不是缺口）
# → 输入恰好落在两条规则的夹缝中，没有任何终结规则被完整触发
initial_D = {"花瓣偏尖", "花朵聚集成一簇"}
facts_D, trace_D = forward_chain(rules, initial_D)
print_results(facts_D, trace_D, "案例 D：规则盲区案例（尖形 + 成簇）")

=== 案例 D：规则盲区案例（尖形 + 成簇） ===
推理轨迹：
  触发 R5: ['花瓣偏尖'] → 尖形花瓣
  触发 R8: ['花朵聚集成一簇'] → 成簇开放

最终结论：（无结论——规则覆盖盲区）



**案例 D 分析：沉默的空结论——CWA 的另一种失效**

系统推导出了两个中间层事实（尖形花瓣、成簇开放），但没有任何终结规则被完全触发，结论为空。

**为什么落入了盲区？**

这朵花的特征组合恰好夹在两条规则之间：

| 规则 | 需要 | 这朵花有 | 差了什么 |
|---|---|---|---|
| R14 桃花 | 尖形花瓣 + **贴枝开放** | 尖形花瓣 | 花是成簇，不是贴枝 |
| R11 樱花 | **缺口花瓣** + 成簇开放 | 成簇开放 | 花瓣是尖形，不是缺口 |

没有规则能完整覆盖"尖形花瓣 + 成簇开放"这个组合，系统于是静默退出。

**这与案例 C 的失效方式截然不同**：

| | 案例 C（分布外） | 案例 D（规则盲区）|
|---|---|---|
| 系统输出 | 自信的错误答案 | 空结论 |
| 用户感知 | 以为系统推理完成，接受错误结论 | 知道没有答案，但不知道是规则不够还是花真的无法识别 |
| 危险程度 | 更危险（主动误导） | 较安全，但无用 |

**两种失效都是 CWA 的表现**：系统无法区分"规则库不够完整"和"输入本身有问题"。  
它不会说"我的知识库可能没有覆盖这种情况"——它只是停下来，或者给出一个答案，且两种情况下语气没有任何差别。

---

## 总结：封闭世界假设的三种失效模式

| 失效模式 | 触发条件 | 系统行为 | 对应 CWA 缺陷 |
|---|---|---|---|
| **危险的沉默失败**（案例 B）| 输入同时满足多条终结规则 | 报告多个结论，无置信度区分 | 无法表达不确定性 |
| **自信的错误答案**（案例 C）| 输入含矛盾特征，某条规则被部分激活 | 给出错误结论，语气与正确答案相同 | 矛盾证据被静默丢弃 |
| **沉默的空结论**（案例 D）| 输入落在所有规则的覆盖空白处 | 无结论输出，不说明原因 | 无法区分"规则不全"与"真的无解" |

**根本问题**：这三种失效有一个共同来源——系统没有关于"自身局限性"的元知识。  
它不知道自己的规则库覆盖了多少，不知道某个输入距离它的"能力边界"有多远，也不知道如何把这种不确定性传递给用户。

这正是第3章之后统计学习方法逐步取代符号推理的数学根源：  
神经网络学到的不只是"规则"，还学到了特征空间中的**连续距离**——陌生输入距离已知样本有多远，这个距离本身就是不确定性的信号。